## 3. Association Rule Mining  (15 points): 

In [1]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("mobile_price.csv")

In [2]:
# 2. Filter the dataset to select only samples where "price_range" = 1
df_filtered = df[df['price_range'] == 1].copy()

In [3]:
# 3. Focus on the specified feature columns
features = ['ram', 'int_memory', 'px_width', 'battery_power']
df_features = df_filtered[features].copy()

In [4]:
# 4. Categorize values into low, medium, and high based on the (max - min) range
def categorize_value(val, min_val, max_val, col_name):
    val_range = max_val - min_val
    
    # Calculate thresholds using the 3:4:3 ratio
    low_thresh = min_val + 0.3 * val_range
    med_thresh = min_val + 0.7 * val_range
    
    prefix = col_name
    
    if val <= low_thresh:
        return f"{prefix}_low"
    elif val <= med_thresh:
        return f"{prefix}_medium"
    else:
        return f"{prefix}_high"

# Create a new DataFrame to hold the categorized data
df_categorized = pd.DataFrame()

# Apply the categorization logic to each column
for col in features:
    col_min = df_features[col].min()
    col_max = df_features[col].max()
    
    df_categorized[col] = df_features[col].apply(
        lambda x: categorize_value(x, col_min, col_max, col)
    )

In [5]:
# 5. Convert each categorized data sample into a transaction format
# This creates a list of lists, where each inner list represents a transaction
transactions = df_categorized.values.tolist()

from IPython.display import display

# Print out the total transactions to verify the results
print(f"Total transactions generated: {len(transactions)}\n")
print("First 5 transaction records:")

# Create a DataFrame for the first 5 transactions for tabular display
# Using the 'features' list from earlier as column headers
df_transactions_display = pd.DataFrame(transactions[:5], columns=features)

# Insert a "Transaction ID" column at the beginning for clarity
df_transactions_display.insert(0, 'Transaction', [f"Transaction {i+1}" for i in range(5)])

# Display the DataFrame as a formatted table in Jupyter Notebook
display(df_transactions_display)

Total transactions generated: 500

First 5 transaction records:


,Transaction,ram,int_memory,px_width,battery_power
0,Transaction 1,ram_high,int_memory_low,px_width_low,battery_power_low
1,Transaction 2,ram_medium,int_memory_medium,px_width_medium,battery_power_high
2,Transaction 3,ram_low,int_memory_medium,px_width_high,battery_power_high
3,Transaction 4,ram_medium,int_memory_medium,px_width_low,battery_power_high
4,Transaction 5,ram_medium,int_memory_high,px_width_low,battery_power_medium


In [6]:
# Print out the first 5 transactions to verify the results
for i, t in enumerate(transactions[:5]):
    print(f"Transaction {i+1}: {', '.join(t)}")

Transaction 1: ram_high, int_memory_low, px_width_low, battery_power_low
Transaction 2: ram_medium, int_memory_medium, px_width_medium, battery_power_high
Transaction 3: ram_low, int_memory_medium, px_width_high, battery_power_high
Transaction 4: ram_medium, int_memory_medium, px_width_low, battery_power_high
Transaction 5: ram_medium, int_memory_high, px_width_low, battery_power_medium


In [7]:
import mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from IPython.display import display

# 1. Initialize the TransactionEncoder
te = TransactionEncoder()

# 2. Fit and transform the transaction records into a one-hot encoded boolean array
te_ary = te.fit(transactions).transform(transactions)

# 3. Convert the array into a pandas DataFrame using the extracted column names
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

# 4. Apply the FP-growth algorithm to find frequent patterns
# We set min_support=0.3 as requested, and use_colnames=True for readable item names
frequent_patterns = fpgrowth(df_onehot, min_support=0.3, use_colnames=True)

# Sort the results by support in descending order for better readability
frequent_patterns = frequent_patterns.sort_values(by='support', ascending=False).reset_index(drop=True)

# 5. Display the results
print(f"Total frequent patterns found (Support >= 0.3): {len(frequent_patterns)}\n")
display(frequent_patterns)

Total frequent patterns found (Support >= 0.3): 8



,support,itemsets
0,0.682,(ram_medium)
1,0.416,(px_width_medium)
2,0.414,(battery_power_medium)
3,0.412,(int_memory_medium)
4,0.318,"(battery_power_medium, ram_medium)"
5,0.316,(int_memory_low)
6,0.308,(battery_power_low)
7,0.306,"(px_width_medium, ram_medium)"


In [8]:
from mlxtend.frequent_patterns import association_rules

# 1. Generate the association rules from the frequent itemsets
# We set the initial metric to 'confidence' with a minimum threshold of 0.4
rules = association_rules(frequent_patterns, metric="confidence", min_threshold=0.4)

# 2. Filter the rules to strictly meet all the requested constraints:
# support >= 0.3, confidence >= 0.4, and lift >= 0.8
# (Note: support >= 0.3 is technically already satisfied by our frequent_patterns, 
# but we include it here to be robust and explicitly follow the instructions)
filtered_rules = rules[
    (rules['support'] >= 0.3) & 
    (rules['confidence'] >= 0.4) & 
    (rules['lift'] >= 0.8)
].copy()

# Sort the rules by confidence and lift for better readability
filtered_rules = filtered_rules.sort_values(by=['confidence', 'lift'], ascending=[False, False]).reset_index(drop=True)

# 3. Display the final filtered association rules
print(f"Total association rules found: {len(filtered_rules)}\n")
display(filtered_rules)

Total association rules found: 4



,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(battery_power_medium),(ram_medium),0.414,0.682,0.318,0.768116,1.126270,1.0,0.035652,1.371375,0.191319,0.408740,0.270805,0.617196
1,(px_width_medium),(ram_medium),0.416,0.682,0.306,0.735577,1.078559,1.0,0.022288,1.202618,0.124720,0.386364,0.168481,0.592129
2,(ram_medium),(battery_power_medium),0.682,0.414,0.318,0.466276,1.126270,1.0,0.035652,1.097945,0.352557,0.408740,0.089208,0.617196
3,(ram_medium),(px_width_medium),0.682,0.416,0.306,0.448680,1.078559,1.0,0.022288,1.059277,0.229046,0.386364,0.055960,0.592129
